In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import random
import time
import sys
import os
import requests  
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import logging

# Import functions from Chatbot module (LLM only)
try:
    from Chatbot import (
        assessment_questions, 
        get_stress_level_with_ollama_api,
        get_personalized_advice_with_ollama,
        model_mapping
    )
    print("✓ Successfully imported REAL Chatbot module functions")
    BACKEND_AVAILABLE = True
    print("✓ LLM-only backend functions loaded")
    
except ImportError as e:
    print(f"✗ Could not import Chatbot module: {e}")
    BACKEND_AVAILABLE = False

def check_ollama_status():
    """Check if Ollama is running and which models are available"""
    try:
        response = requests.get('http://localhost:11434/api/tags', timeout=5)
        if response.status_code == 200:
            models_data = response.json()
            available_models = [model['name'] for model in models_data.get('models', [])]
            print(f"✓ Ollama is running")
            print(f"✓ Available models: {available_models}")
            return True, available_models
        else:
            print(f"✗ Ollama responded with status {response.status_code}")
            return False, []
    except Exception as e:
        print(f"✗ Cannot connect to Ollama: {e}")
        print("Please start Ollama with: ollama serve")
        return False, []

def generate_balanced_test_responses(target_stress_level=None):
    """Generate test responses for target stress level (no manual validation needed)"""
    
    if target_stress_level is None:
        target_stress_level = random.choice([1, 2, 3, 4, 5])
    
    simulated_responses = []
    
    # Generate mood based on target stress level
    if target_stress_level == 1:
        mood = random.choice([" Very good", " Good"])
    elif target_stress_level == 2:
        mood = random.choice([" Good", " Neutral"])
    elif target_stress_level == 3:
        mood = random.choice([" Neutral", " Bad"])
    elif target_stress_level == 4:
        mood = random.choice([" Bad", " Very bad"])
    else:  # target_stress_level == 5
        mood = " Very bad"
    
    simulated_responses.append(mood)

    # Generate responses for the remaining 9 questions based on stress level
    if target_stress_level == 1:  # MINIMAL STRESS
        simulated_responses.append(random.choices(["Yes", "No"], weights=[85, 15])[0])
        
        # Sleep, energy, appetite
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[0, 5, 20, 35, 40])[0])  # Sleep
        simulated_responses.append(random.choices(["Very low", "Low", "Moderate", "High", "Very high"], 
                                                weights=[0, 5, 20, 35, 40])[0])  # Energy
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[0, 5, 20, 35, 40])[0])  # Appetite
        
        # Concentration
        simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[0, 5, 20, 35, 40])[0])
        
        # Overwhelmed
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[50, 35, 15, 0, 0])[0])
        
        # Future outlook
        simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                weights=[0, 0, 15, 40, 45])[0])
        
        # Support
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"],
                                                weights=[0, 5, 15, 35, 45])[0])
        
        # Life thoughts
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[90, 10, 0, 0, 0])[0])
        
    elif target_stress_level == 2:  # LOW STRESS
        simulated_responses.append(random.choices(["Yes", "No"], weights=[70, 30])[0])
        
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[2, 10, 25, 28, 35])[0])  # Sleep
        simulated_responses.append(random.choices(["Very low", "Low", "Moderate", "High", "Very high"], 
                                                weights=[2, 10, 25, 28,35])[0])  # Energy
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[2, 10, 25, 28,35])[0])  # Appetite
        
        simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[5, 15, 25, 30, 25])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[30, 40, 25, 5, 0])[0])
        
        simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                weights=[0, 8, 20, 42, 30])[0])
        
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"],
                                                weights=[2, 10, 20, 35, 33])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[80, 15, 5, 0, 0])[0])
        
    elif target_stress_level == 3:  # MODERATE STRESS
        simulated_responses.append(random.choices(["Yes", "No"], weights=[45, 55])[0])
        
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[15, 30, 35, 15, 5])[0])  # Sleep
        simulated_responses.append(random.choices(["Very low", "Low", "Moderate", "High", "Very high"], 
                                                weights=[15, 30, 35, 15, 5])[0])  # Energy
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[15, 30, 35, 15, 5])[0])  # Appetite
        
        simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[20, 30, 35, 12, 3])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[10, 20, 45, 20, 5])[0])
        
        simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                weights=[8, 25, 40, 22, 5])[0])
        
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"],
                                                weights=[15, 25, 35, 20, 5])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[55, 30, 12, 3, 0])[0])
        
    elif target_stress_level == 4:  # HIGH STRESS
        simulated_responses.append(random.choices(["Yes", "No"], weights=[20, 80])[0])
        
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[35, 40, 20, 4, 1])[0])  # Sleep
        simulated_responses.append(random.choices(["Very low", "Low", "Moderate", "High", "Very high"], 
                                                weights=[35, 40, 20, 4, 1])[0])  # Energy
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[35, 40, 20, 4, 1])[0])  # Appetite
        
        simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[35, 40, 20, 4, 1])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[5, 10, 20, 40, 25])[0])
        
        simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                weights=[30, 40, 22, 6, 2])[0])
        
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"],
                                                weights=[30, 35, 25, 8, 2])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[35, 35, 25, 4, 1])[0])
        
    else:  # target_stress_level == 5: SEVERE STRESS
        simulated_responses.append(random.choices(["Yes", "No"], weights=[5, 95])[0])
        
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[55, 35, 8, 1, 1])[0])  # Sleep
        simulated_responses.append(random.choices(["Very low", "Low", "Moderate", "High", "Very high"], 
                                                weights=[55, 35, 8, 1, 1])[0])  # Energy
        simulated_responses.append(random.choices(["Very poor", "Poor", "Average", "Good", "Very good"], 
                                                weights=[55, 35, 8, 1, 1])[0])  # Appetite
        
        simulated_responses.append(random.choices(["Not at all", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[55, 35, 8, 1, 1])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[2, 5, 15, 35, 43])[0])
        
        simulated_responses.append(random.choices(["Very negative", "Negative", "Neutral", "Positive", "Very positive"], 
                                                weights=[50, 35, 12, 2, 1])[0])
        
        simulated_responses.append(random.choices(["Not at all", "A little", "Somewhat", "Mostly", "Completely"],
                                                weights=[45, 35, 15, 3, 2])[0])
        
        simulated_responses.append(random.choices(["Never", "Rarely", "Sometimes", "Often", "Always"], 
                                                weights=[15, 20, 35, 20, 10])[0])
        
    return simulated_responses, target_stress_level

def is_prediction_acceptable(target, prediction, tolerance=1):
    """Check if prediction is within acceptable range of target"""
    if prediction is None:
        return False
    return abs(target - prediction) <= tolerance

def evaluate_real_backend_model(model_key, num_samples=20, tolerance=1):
    """Evaluate using REAL backend LLM functions ONLY with tolerance"""
    
    if not BACKEND_AVAILABLE:
        print(f"✗ Backend not available, skipping {model_key}")
        return None
    
    model_name = model_mapping[model_key]
    print(f"\n🔄 Evaluating {model_name} using LLM ONLY...")
    print(f"📌 Target model: {model_key} → {model_name}")
    print(f"📏 Tolerance: ±{tolerance} stress level")
    
    # Check if Ollama is available
    ollama_available, available_models = check_ollama_status()
    if not ollama_available:
        print(f"⚠️  Ollama not available - evaluation will fail")
        return None
    
    # Verify the target model is actually available
    if model_name not in available_models:
        print(f"❌ Target model {model_name} not available in Ollama!")
        print(f"Available models: {available_models}")
        return None
    
    # Generate balanced test cases
    stress_levels_to_test = []
    samples_per_level = num_samples // 5
    for level in range(1, 6):
        stress_levels_to_test.extend([level] * samples_per_level)
    
    remaining = num_samples - len(stress_levels_to_test)
    stress_levels_to_test.extend([random.randint(1, 5) for _ in range(remaining)])
    random.shuffle(stress_levels_to_test)
    
    # Initialize tracking
    stress_predictions = []
    target_stress_levels = []
    response_times = []
    successful_predictions = 0
    failed_predictions = 0
    acceptable_predictions = 0  # Within tolerance
    
    print(f"Testing {num_samples} samples with LLM ONLY...")
    
    for i, target_level in enumerate(stress_levels_to_test):
        test_responses, _ = generate_balanced_test_responses(target_level)
        
        try:
            # Use ONLY LLM function - no manual assessment
            print(f"Sample {i+1}/{num_samples}: Calling {model_name} for target level {target_level}...")
            start_time = time.time()
            
            # This calls the actual Ollama API
            pred_stress = get_stress_level_with_ollama_api(test_responses, model_key)
            
            end_time = time.time()
            response_time = end_time - start_time
            
            if pred_stress is not None:
                response_times.append(response_time)
                stress_predictions.append(pred_stress)
                target_stress_levels.append(target_level)
                successful_predictions += 1
                
                # Check if prediction is acceptable (within tolerance)
                is_acceptable = is_prediction_acceptable(target_level, pred_stress, tolerance)
                if is_acceptable:
                    acceptable_predictions += 1
                    status = "✓ ACCEPTABLE"
                else:
                    status = "⚠ OUT_OF_RANGE"
                
                print(f"{status} Sample {i+1}: Target={target_level}, LLM_Pred={pred_stress}, Diff={abs(target_level-pred_stress)}, Time={response_time:.2f}s")
            else:
                failed_predictions += 1
                print(f"✗ Sample {i+1}: LLM prediction failed")
                
        except Exception as e:
            failed_predictions += 1
            print(f"✗ Sample {i+1}: Error - {e}")
    
    # Calculate metrics using target levels as ground truth
    if len(stress_predictions) > 0:
        # Exact accuracy
        exact_accuracy = accuracy_score(target_stress_levels, stress_predictions)
        
        # Tolerance-based accuracy
        tolerance_accuracy = acceptable_predictions / len(stress_predictions)
        
        errors = [abs(p - t) for p, t in zip(stress_predictions, target_stress_levels)]
        avg_error = sum(errors) / len(errors)
        avg_time = sum(response_times) / len(response_times) if response_times else 0
        
        # Calculate precision for each level (exact)
        precision_per_level = {}
        for level in range(1, 6):
            level_preds = [1 if p == level else 0 for p in stress_predictions]
            level_true = [1 if t == level else 0 for t in target_stress_levels]
            if sum(level_preds) > 0:
                precision_per_level[level] = precision_score(level_true, level_preds, zero_division=0)
            else:
                precision_per_level[level] = 0.0
        
        avg_precision = sum(precision_per_level.values()) / len(precision_per_level)
        
        # Depression classification (stress >= 4)
        depression_preds = [p >= 4 for p in stress_predictions]
        depression_true = [t >= 4 for t in target_stress_levels]
        
        depression_accuracy = accuracy_score(depression_true, depression_preds)
        depression_f1 = f1_score(depression_true, depression_preds, zero_division=0)
        
        print(f"\n📊 {model_name} Results (LLM ONLY):")
        print(f"Success Rate: {successful_predictions}/{num_samples} ({successful_predictions/num_samples*100:.1f}%)")
        print(f"Failed Predictions: {failed_predictions}")
        print(f"Exact Accuracy: {exact_accuracy:.3f}")
        print(f"Tolerance Accuracy (±{tolerance}): {tolerance_accuracy:.3f}")
        print(f"Acceptable Predictions: {acceptable_predictions}/{successful_predictions}")
        print(f"Average Error: {avg_error:.3f}")
        print(f"Average Precision: {avg_precision:.3f}")
        print(f"Average Response Time: {avg_time:.2f}s")
        print(f"Depression F1: {depression_f1:.3f}")
        
        return {
            'model_key': model_key,
            'model_name': model_name,
            'success_rate': successful_predictions / num_samples,
            'failed_predictions': failed_predictions,
            'exact_accuracy': exact_accuracy,
            'tolerance_accuracy': tolerance_accuracy,
            'acceptable_predictions': acceptable_predictions,
            'stress_error': avg_error,
            'stress_precision': avg_precision,
            'response_time': avg_time,
            'depression_f1': depression_f1,
            'predictions': stress_predictions,
            'true_values': target_stress_levels
        }
    
    return None

def run_real_backend_evaluation(num_samples=25, tolerance=1):
    """Run evaluation using REAL backend LLM functions with tolerance"""
    start_time = time.time()
    
    print("🚀 === MindCare-AI LLM-ONLY Backend Evaluation ===")
    print("⚠️  This will take time because it calls real LLM models!")
    print(f"⏱️  Expected time: ~{num_samples * len(model_mapping) * 3} seconds")
    print(f"📏 Tolerance: ±{tolerance} stress level\n")
    
    if not BACKEND_AVAILABLE:
        print("✗ Backend functions not available!")
        return None
    
    # Check Ollama first
    ollama_available, available_models = check_ollama_status()
    if not ollama_available:
        print("❌ Ollama is not running! Please start it with: ollama serve")
        return None
    
    results = []
    
    for model_key in model_mapping:
        print(f"\n{'='*60}")
        print(f"🔍 Evaluating {model_mapping[model_key]}...")
        print(f"{'='*60}")
        
        result = evaluate_real_backend_model(model_key, num_samples, tolerance)
        if result:
            results.append(result)
        else:
            print(f"❌ Failed to evaluate {model_key}")
    
    if not results:
        print("❌ No models were successfully evaluated!")
        return None
    
    # Create comparison
    comparison_data = []
    for r in results:
        comparison_data.append({
            'Model': r['model_name'],
            'Success Rate': f"{r['success_rate']:.1%}",
            'Failed Predictions': r['failed_predictions'],
            'Exact Accuracy': r['exact_accuracy'],
            f'Tolerance Accuracy (±{tolerance})': r['tolerance_accuracy'],
            'Acceptable Preds': r['acceptable_predictions'],
            'Avg Error': r['stress_error'],
            'Avg Time (s)': r['response_time'],
            'Depression F1': r['depression_f1']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print(f"\n{'='*80}")
    print("📈 LLM-ONLY EVALUATION RESULTS")
    print(f"{'='*80}")
    print(comparison_df.to_string(index=False))
    
    end_time = time.time()
    total_time = end_time - start_time
    print(f"\n⏱️  Total evaluation time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
    
    # Create visualization
    if len(results) > 1:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        models = [r['model_name'] for r in results]
        
        # Exact vs Tolerance Accuracy
        exact_accuracy = [r['exact_accuracy'] for r in results]
        tolerance_accuracy = [r['tolerance_accuracy'] for r in results]
        
        x = np.arange(len(models))
        width = 0.35
        
        axes[0, 0].bar(x - width/2, exact_accuracy, width, label='Exact', alpha=0.8)
        axes[0, 0].bar(x + width/2, tolerance_accuracy, width, label=f'±{tolerance}', alpha=0.8)
        axes[0, 0].set_title('Accuracy Comparison')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].set_xticks(x)
        axes[0, 0].set_xticklabels(models, rotation=45)
        axes[0, 0].legend()
        
        # Response time
        times = [r['response_time'] for r in results]
        axes[0, 1].bar(models, times)
        axes[0, 1].set_title('Average Response Time')
        axes[0, 1].set_ylabel('Time (seconds)')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        # Success rate
        success_rates = [r['success_rate'] for r in results]
        axes[1, 0].bar(models, success_rates)
        axes[1, 0].set_title('Success Rate')
        axes[1, 0].set_ylabel('Success Rate')
        axes[1, 0].tick_params(axis='x', rotation=45)
        
        # Average Error
        errors = [r['stress_error'] for r in results]
        axes[1, 1].bar(models, errors)
        axes[1, 1].set_title('Average Prediction Error')
        axes[1, 1].set_ylabel('Average Error')
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig('llm_only_evaluation.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    return comparison_df

# Test function - simplified without manual assessment
def test_real_backend():
    """Test that we're using real backend LLM functions"""
    if not BACKEND_AVAILABLE:
        print("❌ Backend not available")
        return False
    
    print("🧪 Testing real backend LLM functions...")
    
    # Test sample
    test_responses = [
        " Very bad",  # Mood
        "No",        # Enjoyment
        "Very poor", # Sleep
        "Very poor", # Energy  
        "Poor",      # Appetite
        "Not at all", # Concentration
        "Always",    # Overwhelmed
        "Very negative", # Outlook
        "Not at all",   # Support
        "Often"         # Life thoughts
    ]
    
    try:
        print("Testing LLM assessment (this should take several seconds)...")
        start = time.time()
        llm_result = get_stress_level_with_ollama_api(test_responses, 'gemma')
        end = time.time()
        print(f"✓ LLM assessment: {llm_result} (took {end-start:.2f}s)")
        
        if end - start > 1:  # Should take at least 1 second for real API call
            print("✅ REAL LLM backend functions are working!")
            return True
        else:
            print("⚠️  Response too fast - might be an issue")
            return False
            
    except Exception as e:
        print(f"❌ Test failed: {e}")
        return False

print("\n🎯 === LLM-ONLY Model Evaluation Ready ===")
print("Available functions:")
print("- test_real_backend() - Test that LLM functions work")
print("- run_real_backend_evaluation(num_samples, tolerance) - Run LLM-ONLY evaluation")
print("📏 Tolerance default: ±1 stress level")
print("✅ Manual assessment completely removed!")

if BACKEND_AVAILABLE:
    print("✅ LLM-only backend functions loaded")
else:
    print("❌ Backend functions not available")

✓ Successfully imported REAL Chatbot module functions
✓ LLM-only backend functions loaded

🎯 === LLM-ONLY Model Evaluation Ready ===
Available functions:
- test_real_backend() - Test that LLM functions work
- run_real_backend_evaluation(num_samples, tolerance) - Run LLM-ONLY evaluation
📏 Tolerance default: ±1 stress level
✅ Manual assessment completely removed!
✅ LLM-only backend functions loaded


In [5]:
test_real_backend()

INFO:Chatbot:Trying stress assessment with selected model: gemma3:4b


🧪 Testing real backend LLM functions...
Testing LLM assessment (this should take several seconds)...


INFO:Chatbot:Model gemma3:4b raw response: '5'
INFO:Chatbot:✓ Successfully extracted stress level 5 from gemma3:4b


✓ LLM assessment: 5 (took 7.98s)
✅ REAL LLM backend functions are working!


True

In [ ]:
results = run_real_backend_evaluation(num_samples=20, tolerance=1)

🚀 === MindCare-AI LLM-ONLY Backend Evaluation ===
⚠️  This will take time because it calls real LLM models!
⏱️  Expected time: ~240 seconds
📏 Tolerance: ±1 stress level

✓ Ollama is running
✓ Available models: ['qwen3:8b', 'deepseek-r1:7b', 'llama3.1:8b', 'gemma3:4b']

🔍 Evaluating gemma3:4b...

🔄 Evaluating gemma3:4b using LLM ONLY...
📌 Target model: gemma → gemma3:4b
📏 Tolerance: ±1 stress level


INFO:Chatbot:Trying stress assessment with selected model: gemma3:4b


✓ Ollama is running
✓ Available models: ['qwen3:8b', 'deepseek-r1:7b', 'llama3.1:8b', 'gemma3:4b']
Testing 20 samples with LLM ONLY...
Sample 1/20: Calling gemma3:4b for target level 5...


INFO:Chatbot:Model gemma3:4b raw response: '4'
INFO:Chatbot:✓ Successfully extracted stress level 4 from gemma3:4b
INFO:Chatbot:Trying stress assessment with selected model: gemma3:4b


✓ ACCEPTABLE Sample 1: Target=5, LLM_Pred=4, Diff=1, Time=3.37s
Sample 2/20: Calling gemma3:4b for target level 4...


...

✓ ACCEPTABLE Sample 19: Target=3, LLM_Pred=4, Diff=1, Time=2.94s
Sample 20/20: Calling llama3.1:8b for target level 5...


INFO:Chatbot:Model llama3.1:8b raw response: '4'
INFO:Chatbot:✓ Successfully extracted stress level 4 from llama3.1:8b


✓ ACCEPTABLE Sample 20: Target=5, LLM_Pred=4, Diff=1, Time=3.03s

📊 llama3.1:8b Results (LLM ONLY):
Success Rate: 20/20 (100.0%)
Failed Predictions: 0
Exact Accuracy: 0.150
Tolerance Accuracy (±1): 0.950
Acceptable Predictions: 19/20
Average Error: 0.900
Average Precision: 0.107
Average Response Time: 3.64s
Depression F1: 0.762

🔍 Evaluating qwen3:8b...

🔄 Evaluating qwen3:8b using LLM ONLY...
📌 Target model: qwen3 → qwen3:8b
📏 Tolerance: ±1 stress level


INFO:Chatbot:Trying stress assessment with selected model: qwen3:8b


✓ Ollama is running
✓ Available models: ['qwen3:8b', 'deepseek-r1:7b', 'llama3.1:8b', 'gemma3:4b']
Testing 20 samples with LLM ONLY...
Sample 1/20: Calling qwen3:8b for target level 5...
